In [4]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [119]:
movies

,item_id,tittle
0,1,Toy Story (1995)
1,2,GoldenEye (1995)
2,3,Four Rooms (1995)
3,4,Get Shorty (1995)
4,5,Copycat (1995)
...,...,...
1677,1678,Mat' i syn (1997)
1678,1679,B. Monkey (1998)
1679,1680,Sliding Doors (1998)
1680,1681,You So Crazy (1994)


In [121]:
ratings

,user_id,item_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596
...,...,...,...,...
99995,880,476,3,880175444
99996,716,204,5,879795543
99997,276,1090,1,874795795
99998,13,225,2,882399156


In [7]:
ratings = pd.read_csv("u.data",sep="\t",names= ['user_id', 'item_id', 'rating', 'timestamp'])
movies = pd.read_csv("u.item",sep="|",encoding="latin-1",header =None)
movies = movies[[0,1]]
movies.columns = ["item_id","tittle"]
movies_df = pd.merge(ratings,movies,on="item_id")
movies_df["timestamp"] = pd.to_datetime(movies_df["timestamp"],unit ="s")
movies_df

,user_id,item_id,rating,timestamp,tittle
0,196,242,3,1997-12-04 15:55:49,Kolya (1996)
1,186,302,3,1998-04-04 19:22:22,L.A. Confidential (1997)
2,22,377,1,1997-11-07 07:18:36,Heavyweights (1994)
3,244,51,2,1997-11-27 05:02:03,Legends of the Fall (1994)
4,166,346,1,1998-02-02 05:33:16,Jackie Brown (1997)
...,...,...,...,...,...
99995,880,476,3,1997-11-22 05:10:44,"First Wives Club, The (1996)"
99996,716,204,5,1997-11-17 19:39:03,Back to the Future (1985)
99997,276,1090,1,1997-09-20 22:49:55,Sliver (1993)
99998,13,225,2,1997-12-17 22:52:36,101 Dalmatians (1996)


##  Data understanding

In [3]:
movies_df.isna().sum()

user_id      0
item_id      0
rating       0
timestamp    0
tittle       0
dtype: int64

In [4]:
movies_df.dtypes

user_id               int64
item_id               int64
rating                int64
timestamp    datetime64[ns]
tittle               object
dtype: object

# Modul building

## Model popularity

In [106]:
def model_popularity(top_n=5):
    pop = movies_df.groupby("tittle").agg({"rating":["mean","count"]})
    pop.columns = ["Average_rating","count"]
    pop = pop[pop["count"]>50]
    pop = pop.sort_values(by =["Average_rating"],ascending= False )
    return pop.head().index.tolist()

In [107]:
model_popularity()

['Close Shave, A (1995)',
 "Schindler's List (1993)",
 'Wrong Trousers, The (1993)',
 'Casablanca (1942)',
 'Wallace & Gromit: The Best of Aardman Animation (1996)']

## Model current trend

In [108]:
def model_trending(top_n=5):
    recent = movies_df[movies_df["timestamp"]> '1998-01-01']
    trend= recent.groupby("tittle")["rating"].agg(["mean","count"])
    trend.columns = ["Average_mean","count"]
    trend = trend.sort_values(by=["count"],ascending=False)
    return trend.head().index.tolist()

In [109]:
model_trending()

['Titanic (1997)',
 'Contact (1997)',
 'Star Wars (1977)',
 'Air Force One (1997)',
 'Scream (1996)']

# Content model

In [94]:
cv = CountVectorizer(stop_words="english")
cv_movies = cv.fit_transform(movies["tittle"])
content_corr = cosine_similarity(cv_movies)

In [91]:
def model_content(movie_name,top_n=5):
    if movie_name not in movies["tittle"].values:
        return("movie not found")
    idx = movies[movies["tittle"]==movie_name].index[0]
    scores = list(enumerate(content_corr[idx]))
    scores = sorted(scores,key=lambda x: x[1],reverse=True)[1:top_n+1]
    print(scores)
    return[movies.iloc[i[0]].tittle for i in scores]
    

In [92]:
model_content("Titanic (1997)")

[(300, np.float64(0.7071067811865475)), (1024, np.float64(0.7071067811865475)), (1527, np.float64(0.7071067811865475)), (1537, np.float64(0.7071067811865475)), (1608, np.float64(0.7071067811865475))]


['In & Out (1997)',
 'Fire Down Below (1997)',
 'Nowhere (1997)',
 'All Over Me (1997)',
 'B*A*P*S (1997)']

# model_collobarative

In [81]:
collob_pivot_table = pd.pivot_table(data = movies_df,values="rating",index="user_id",columns="tittle").fillna(0)
collob_corr = cosine_similarity(collob_pivot_table.T)
collob_df = pd.DataFrame(collob_corr,
            index = collob_pivot_table.columns,
            columns = collob_pivot_table.columns)

In [95]:
def model_collob(movie_name,top_n=5):
    if movie_name not in collob_df.columns:
        return("not to found")
    scores = collob_df[movie_name].sort_values(ascending = False)[1:top_n+1]
    return scores.index.tolist() 

In [96]:
model_collob("L.A. Confidential (1997)")

['Full Monty, The (1997)',
 'English Patient, The (1996)',
 'Contact (1997)',
 'Titanic (1997)',
 'Apt Pupil (1998)']

# model_hybrid 

In [98]:
def model_hybrid(movie_name):
    content = model_content(movie_name)
    collob = model_collob(movie_name)
    combined = list(dict.fromkeys(content + collob))
    return combined[:5]

In [101]:
model_hybrid("L.A. Confidential (1997)")

[(300, np.float64(0.7071067811865475)), (1024, np.float64(0.7071067811865475)), (1527, np.float64(0.7071067811865475)), (1537, np.float64(0.7071067811865475)), (1608, np.float64(0.7071067811865475))]


['In & Out (1997)',
 'Fire Down Below (1997)',
 'Nowhere (1997)',
 'All Over Me (1997)',
 'B*A*P*S (1997)']

# Recomended all 

In [117]:
def recommend_all(movie_name):
    print(f"\n🎬 Recommendations for: {movie_name}\n")

    print(" Popular:")
    print(model_popularity())

    print("\n Trending:")
    print(model_trending())

    print("\n Content-Based:")
    print(model_content(movie_name))

    print("\n Collaborative:")
    print(model_collob(movie_name))

    print("\n Hybrid:")
    print(model_hybrid(movie_name))

In [118]:
recommend_all('Fire Down Below (1997)')


🎬 Recommendations for: Fire Down Below (1997)

 Popular:
['Close Shave, A (1995)', "Schindler's List (1993)", 'Wrong Trousers, The (1993)', 'Casablanca (1942)', 'Wallace & Gromit: The Best of Aardman Animation (1996)']

 Trending:
['Titanic (1997)', 'Contact (1997)', 'Star Wars (1977)', 'Air Force One (1997)', 'Scream (1996)']

 Content-Based:
[(1024, np.float64(1.0)), (1527, np.float64(1.0)), (1537, np.float64(1.0)), (1608, np.float64(1.0)), (242, np.float64(0.7071067811865475))]
['Fire Down Below (1997)', 'Nowhere (1997)', 'All Over Me (1997)', 'B*A*P*S (1997)', 'Jungle2Jungle (1997)']

 Collaborative:
['187 (1997)', 'G.I. Jane (1997)', 'Free Willy 3: The Rescue (1997)', 'Murder at 1600 (1997)', 'Edge, The (1997)']

 Hybrid:
[(1024, np.float64(1.0)), (1527, np.float64(1.0)), (1537, np.float64(1.0)), (1608, np.float64(1.0)), (242, np.float64(0.7071067811865475))]
['Fire Down Below (1997)', 'Nowhere (1997)', 'All Over Me (1997)', 'B*A*P*S (1997)', 'Jungle2Jungle (1997)']


In [74]:
movies_df.head()

,user_id,item_id,rating,timestamp,tittle
0,196,242,3,1997-12-04 15:55:49,Kolya (1996)
1,186,302,3,1998-04-04 19:22:22,L.A. Confidential (1997)
2,22,377,1,1997-11-07 07:18:36,Heavyweights (1994)
3,244,51,2,1997-11-27 05:02:03,Legends of the Fall (1994)
4,166,346,1,1998-02-02 05:33:16,Jackie Brown (1997)
